In [3]:
import pandas as pd
import numpy as np
import sklearn

print("Pandas:", pd.__version__)
print("Ready to roll!")

Pandas: 3.0.5
Ready to roll!


In [4]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sub = pd.read_csv("data/sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape: ", test.shape)
print("Sub shape:  ", sub.shape)

train.head(20)

Train shape: (6818, 13)
Test shape:  (1705, 12)
Sub shape:   (1705, 2)


,id,product_code,product_weight_kg,fat_content,shelf_visibility,product_category,product_price,store_code,store_age_years,store_size,store_location_tier,store_format,total_sales
0,row_00000,PRD-PRFP9S,14.252,Low Fat,0.0271,Frozen Foods,81.37,STORE-AGY,45,Large,Tier_3,Standard Supermarket,1764.98
1,row_00001,PRD-PXXK71,7.698,Low Fat,0.0720,HEALTH AND HYGIENE,42.05,STORE-YLW,35,Small,Tier_1,Standard Supermarket,342.13
2,row_00002,PRD-V5MOIJ,14.264,Regular,0.0421,Canned,41.35,STORE-89Z,33,Medium,Tier_1,Standard Supermarket,378.85
3,row_00003,PRD-UN5Z3J,NaN,Regular,0.0449,soft drinks,174.35,STORE-7WS,47,Medium,Tier_3,Flagship Hypermarket,5595.72
4,row_00004,PRD-6RDQYB,10.338,Regular,0.0120,meat,203.06,STORE-9RG,28,Small,Tier_2,Standard Supermarket,2375.36
5,row_00005,PRD-E6VA05,9.222,Low Fat,0.0405,soft drinks,31.69,STORE-89Z,33,Medium,Tier_1,Standard Supermarket,842.75
6,row_00006,PRD-MHLD93,NaN,Low Fat,0.1239,Canned,212.23,STORE-7WS,47,Medium,Tier_3,Flagship Hypermarket,4512.70
7,row_00007,PRD-EF6Y39,10.210,Regular,0.0133,Snack Foods,142.70,STORE-HL7,23,Medium,Tier_3,Superstore,2373.55
8,row_00008,PRD-IAZMTU,14.054,Low Fat,0.0233,baking goods,102.71,STORE-HL7,23,Medium,Tier_3,Superstore,2037.05
9,row_00010,PRD-DH2TA3,8.755,Low Fat,0.0258,household,104.60,STORE-JOR,34,NaN,Tier_3,Corner Shop,213.93


In [5]:
# Check missing data percentages
missing = train.isnull().sum()
missing_pct = (missing / len(train)) * 100
pd.DataFrame({"missing_count": missing, "percent": missing_pct}).query("missing_count > 0")

,missing_count,percent
product_weight_kg,1225,17.967146
store_size,1919,28.146084


In [6]:
train.describe()

,product_weight_kg,shelf_visibility,product_price,store_age_years,total_sales
count,5593.000000,6818.000000,6818.000000,6818.000000,6818.000000
mean,12.850876,0.065527,140.473623,34.178205,2174.756574
std,4.646542,0.051566,62.413513,8.384574,1697.894956
min,4.482000,0.000000,31.110000,23.000000,32.700000
25%,8.750000,0.026600,93.280000,28.000000,834.792500
50%,12.647000,0.053200,142.065000,33.000000,1790.890000
75%,16.894000,0.093400,185.987500,45.000000,3092.587500
max,21.883000,0.320100,273.040000,47.000000,12996.820000


In [7]:
# Quick look at the target variable distribution
train['total_sales'].describe()

count     6818.000000
mean      2174.756574
std       1697.894956
min         32.700000
25%        834.792500
50%       1790.890000
75%       3092.587500
max      12996.820000
Name: total_sales, dtype: float64

In [8]:
#standardizing categorical columns
cat_cols = ['fat_content', 'product_category', 'store_size', 'store_location_tier', 'store_format']
for c in cat_cols:
    train[c] = train[c].astype(str).str.strip().str.title()
    test[c] = test[c].astype(str).str.strip().str.title()

In [9]:
for col in ['fat_content', 'store_size', 'store_location_tier', 'store_format']:
    print(f"--- {col} ---")
    print(train[col].value_counts(dropna=False), "\n")

--- fat_content ---
fat_content
Low Fat    4425
Regular    2393
Name: count, dtype: int64 

--- store_size ---
store_size
Medium    2218
Small     1933
NaN       1919
Large      748
Name: count, dtype: int64 

--- store_location_tier ---
store_location_tier
Tier_3    2677
Tier_2    2232
Tier_1    1909
Name: count, dtype: int64 

--- store_format ---
store_format
Standard Supermarket    4462
Corner Shop              866
Flagship Hypermarket     748
Superstore               742
Name: count, dtype: int64 



In [10]:
# Check unique counts vs total rows
print(f"Total rows: {len(train)}")
print(f"Unique product codes: {train['product_code'].nunique()}")


Total rows: 6818
Unique product codes: 1555


In [11]:
# 1. Combine or align to avoid data leakage
# Compute item weights lookup from both train and test to maximize coverage
all_data = pd.concat([train.assign(is_train=1), test.assign(is_train=0, total_sales=np.nan)], ignore_index=True)

# 2. Impute product_weight_kg by product_code
item_weights = all_data.groupby('product_code')['product_weight_kg'].transform('mean')
all_data['product_weight_kg'] = all_data['product_weight_kg'].fillna(item_weights)
all_data['product_weight_kg'] = all_data['product_weight_kg'].fillna(all_data['product_weight_kg'].median())

# 3. Handle 0.0 in shelf_visibility as missing
item_vis = all_data[all_data['shelf_visibility'] > 0].groupby('product_code')['shelf_visibility'].transform('mean')
all_data.loc[all_data['shelf_visibility'] == 0, 'shelf_visibility'] = np.nan
all_data['shelf_visibility'] = all_data['shelf_visibility'].fillna(item_vis)
all_data['shelf_visibility'] = all_data['shelf_visibility'].fillna(all_data['shelf_visibility'].median())

# 4. Impute store_size using mode per store_format
store_size_mode = all_data.groupby('store_format')['store_size'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else 'Small')
all_data['store_size'] = all_data['store_size'].fillna(all_data['store_format'].map(store_size_mode))

# 5. Standardize category strings
all_data['product_category'] = all_data['product_category'].astype(str).str.strip().str.title()

# 6. Separate back into train and test
train_clean = all_data[all_data['is_train'] == 1].drop(columns=['is_train']).copy()
test_clean = all_data[all_data['is_train'] == 0].drop(columns=['is_train', 'total_sales']).copy()

print("Remaining nulls in train:")
print(train_clean.isnull().sum())

Remaining nulls in train:
id                     0
product_code           0
product_weight_kg      0
fat_content            0
shelf_visibility       0
product_category       0
product_price          0
store_code             0
store_age_years        0
store_size             0
store_location_tier    0
store_format           0
total_sales            0
dtype: int64


In [12]:
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

# Features
for df in [train_clean, test_clean]:
    # Product code prefix (often indicates general item type: e.g. 'PRD' or category subgroups)
    df['code_prefix'] = df['product_code'].str[:6]

# Defining feature subsets
num_cols = ['product_weight_kg', 'shelf_visibility', 'product_price', 'store_age_years']
ord_cols = ['store_size', 'store_location_tier']
cat_cols = ['fat_content', 'product_category', 'store_format', 'code_prefix']

# Ordinal mappings
size_order = ['Small', 'Medium', 'Large']
tier_order = ['Tier_1', 'Tier_2', 'Tier_3']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_cols),
        ('ord', OrdinalEncoder(categories=[size_order, tier_order]), ord_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

X = train_clean[num_cols + ord_cols + cat_cols]
y = train_clean['total_sales']
X_test = test_clean[num_cols + ord_cols + cat_cols]

In [26]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    # Fit pipeline
    model_pipeline = Pipeline([
        ('prep', preprocessor),
        ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
    ])
    
    model_pipeline.fit(X_tr, y_tr)
    preds = model_pipeline.predict(X_va)
    
    # Clip negative predictions to zero (sales cannot be negative)
    preds = np.clip(preds, 0, None)
    
    fold_rmse = root_mean_squared_error(y_va, preds)
    rmse_scores.append(fold_rmse)
    print(f"Fold {fold + 1} RMSE: {fold_rmse:.2f}")

print(f"\nMean CV RMSE: {np.mean(rmse_scores):.2f}")

Fold 1 RMSE: 1134.08
Fold 2 RMSE: 1092.88
Fold 3 RMSE: 1152.44
Fold 4 RMSE: 1176.63
Fold 5 RMSE: 1124.95

Mean CV RMSE: 1136.19


In [9]:
# Train baseline on all available training data
baseline_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

baseline_pipeline.fit(X, y)

# Generate test predictions and clip negative values
test_preds = baseline_pipeline.predict(X_test)
test_preds = np.clip(test_preds, 0, None)

# Build submission DataFrame matching sample_submission structure
submission = pd.DataFrame({
    'id': test_clean['id'],
    'total_sales': test_preds
})

submission.to_csv("data/baseline_sub.csv", index=False)
print("Saved data/baseline_sub.csv successfully!")
submission.head()

Saved data/baseline_sub.csv successfully!


,id,total_sales
6818,row_00009,2687.1137
6819,row_00015,6676.3732
6820,row_00019,3335.3678
6821,row_00020,3366.6209
6822,row_00023,2479.1877


In [10]:
submission.shape

(1705, 2)

In [13]:
import lightgbm as lgb
import numpy as np

# 1. Transform the target into log space
y_log = np.log1p(y)

# 2. Setup K-Fold and test prediction accumulator
kf = KFold(n_splits=5, shuffle=True, random_state=42)
lgb_cv_scores = []
oof_preds = np.zeros(len(train_clean))
test_preds_lgb = np.zeros(len(test_clean))

# Prepare transformed features
X_prepped = preprocessor.fit_transform(X)
X_test_prepped = preprocessor.transform(X_test)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_prepped, y_log)):
    X_tr, y_tr_log = X_prepped[train_idx], y_log.iloc[train_idx]
    X_va, y_va_log = X_prepped[val_idx], y_log.iloc[val_idx]
    
    # LightGBM Regressor
    model = lgb.LGBMRegressor(
        n_estimators=300,
        learning_rate=0.03,
        num_leaves=31,
        random_state=42,
        verbosity=-1
    )
    
    # Fit model on log-transformed target
    model.fit(X_tr, y_tr_log)
    
    # Predict validation fold in log space, then invert back to original scale
    val_preds_log = model.predict(X_va)
    val_preds_actual = np.clip(np.expm1(val_preds_log), 0, None)
    oof_preds[val_idx] = val_preds_actual
    
    # Evaluate RMSE on the original sales scale
    fold_rmse = root_mean_squared_error(y.iloc[val_idx], val_preds_actual)
    lgb_cv_scores.append(fold_rmse)
    print(f"Fold {fold + 1} RMSE: {fold_rmse:.2f}")
    
    # Accumulate fold predictions for the test set (OOF ensembling)
    test_fold_log = model.predict(X_test_prepped)
    test_preds_lgb += np.clip(np.expm1(test_fold_log), 0, None) / kf.n_splits

print(f"\nMean LightGBM CV RMSE: {np.mean(lgb_cv_scores):.2f}")

Fold 1 RMSE: 1139.73
Fold 2 RMSE: 1106.76
Fold 3 RMSE: 1127.51
Fold 4 RMSE: 1162.59
Fold 5 RMSE: 1120.18

Mean LightGBM CV RMSE: 1131.36


In [14]:
# Add domain features directly to train and test
for df in [train_clean, test_clean]:
    # Price per unit weight (value density)
    df['price_per_kg'] = df['product_price'] / (df['product_weight_kg'] + 1e-5)
    
    # Store-level aggregations
    store_price_mean = df.groupby('store_format')['product_price'].transform('mean')
    df['price_diff_store_mean'] = df['product_price'] - store_price_mean
    
    # Visibility relative to store average
    vis_store_mean = df.groupby('store_code')['shelf_visibility'].transform('mean')
    df['vis_ratio_store'] = df['shelf_visibility'] / (vis_store_mean + 1e-5)

# Update feature list
num_cols = [
    'product_weight_kg', 'shelf_visibility', 'product_price', 
    'store_age_years', 'price_per_kg', 'price_diff_store_mean', 'vis_ratio_store'
]
ord_cols = ['store_size', 'store_location_tier']
cat_cols = ['fat_content', 'product_category', 'store_format', 'code_prefix', 'store_code']

# Update preprocessor to include store_code
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_cols),
        ('ord', OrdinalEncoder(categories=[size_order, tier_order]), ord_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

X_prepped = preprocessor.fit_transform(train_clean[num_cols + ord_cols + cat_cols])
X_test_prepped = preprocessor.transform(test_clean[num_cols + ord_cols + cat_cols])

In [15]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
lgb_cv_scores = []
oof_preds = np.zeros(len(train_clean))
test_preds_lgb = np.zeros(len(test_clean))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_prepped, y)):
    X_tr, y_tr = X_prepped[train_idx], y.iloc[train_idx]
    X_va, y_va = X_prepped[val_idx], y.iloc[val_idx]
    
    model = lgb.LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=24,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=-1
    )
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=40, verbose=False)]
    )
    
    preds = np.clip(model.predict(X_va), 0, None)
    oof_preds[val_idx] = preds
    
    fold_rmse = root_mean_squared_error(y.iloc[val_idx], preds)
    lgb_cv_scores.append(fold_rmse)
    print(f"Fold {fold + 1} RMSE: {fold_rmse:.2f}")
    
    test_fold = np.clip(model.predict(X_test_prepped), 0, None)
    test_preds_lgb += test_fold / kf.n_splits

print(f"\nMean Tuned LightGBM CV RMSE: {np.mean(lgb_cv_scores):.2f}")

c:\Users\USER\Desktop\ML_zoomcamp-1\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 1 RMSE: 1078.00


c:\Users\USER\Desktop\ML_zoomcamp-1\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 2 RMSE: 1059.94


c:\Users\USER\Desktop\ML_zoomcamp-1\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 3 RMSE: 1091.45


c:\Users\USER\Desktop\ML_zoomcamp-1\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 4 RMSE: 1118.09


c:\Users\USER\Desktop\ML_zoomcamp-1\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 5 RMSE: 1073.10

Mean Tuned LightGBM CV RMSE: 1084.12


In [16]:
sub_lgb = pd.DataFrame({
    'id': test_clean['id'],
    'total_sales': test_preds_lgb
})

sub_lgb.to_csv("data/lgb_tuned_sub.csv", index=False)
print("Saved data/lgb_tuned_sub.csv successfully!")
print(sub_lgb.head())

Saved data/lgb_tuned_sub.csv successfully!
             id  total_sales
6818  row_00009  2911.871217
6819  row_00015  4136.466212
6820  row_00019  3114.300472
6821  row_00020  3017.048809
6822  row_00023  2446.685493
